# GPS Location & EMA Analysis — 30 Participants

**Input file:** `unfiltered_30_participants_daily_features_with_gps.csv`
(one row = one GPS measurement; daily EMA / feature values are repeated on every GPS row of that day)

## Notebook map

| Part | Section | What you get |
|---|---|---|
| **1. Location** | 1.1 | Load data & quick overview |
| | 1.2 | List of all participant IDs |
| | 1.3 | Number of GPS points per participant |
| | 1.4 | Number of days GPS was captured per participant |
| | 1.5 | Day-wise detail: GPS points per day + first GPS timestamp (saved as CSV) |
| | 1.6 | Outlier detection & cleaning (4 methods, per-participant summary, cleaned CSVs) |
| | 1.7 | Geomap per participant of the **cleaned** points, coloured by Home / Work / Other |
| **2. EMA** | 2.1 | What counts as an EMA record in this file |
| | 2.2 | EMA count and number of EMA days per participant |
| | 2.3 | Day-wise EMA detail with times (`HH:MM`, comma separated) |

## How to run
1. Put this notebook and the CSV in the **same folder** (or edit the paths in the settings cell below).
2. Run **Kernel → Restart & Run All**. Sections run top-to-bottom; nothing needs to be edited unless you want to.
3. Files created by the notebook are saved next to it (listed at the end of each section).

Required libraries: `pandas`, `numpy`, `plotly` (`pip install pandas numpy plotly`)

## ⚙️ Settings & imports
All the things you might want to change are in this one cell.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
from math import radians, sin, cos, sqrt, atan2

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# ── File paths ─────────────────────────────────────────────────────────────
DATA_FILE = "unfiltered_30_participants_daily_features_with_gps.csv"
CLEANED_FILE_FOR_MAP = "gps_outliers_removed.csv"

# ── Outlier thresholds ─────────────────────────────────────────────────────
IQR_MULTIPLIER = 3.0
MAX_SPEED_MS = 35.0
MAX_SPEED_ACCURACY = 5.0

# IMPORTANT: location cleaning and speed-quality flags are kept separate.
# Only impossible coordinates + participant-specific coordinate IQR outliers
# are removed from the cleaned GPS CSV. Speed flags remain diagnostic columns.

# ── Map settings ───────────────────────────────────────────────────────────
LABEL_COLORS = {"Home": "#2ca02c", "Work": "#1f77b4", "Other": "#ff7f0e"}
PARTICIPANTS_TO_PLOT = None

# ── EMA settings ───────────────────────────────────────────────────────────
EMA_SCORE_COLS = ["pos_emotion", "neg_emotion"]
EMA_TIMESTAMP_COL = None


---
# PART 1 — LOCATION (GPS) DATA
---
## 1.1 Load the data
Reads the CSV and builds one proper `timestamp` column from `date` + `time`.

In [ ]:
df = pd.read_csv(DATA_FILE, low_memory=False)
df["timestamp"] = pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str), format="mixed")

print(f"Rows (GPS points): {len(df):,}")
print(f"Columns          : {df.shape[1]}")
print(f"Participants     : {df['participant_id'].nunique()}")
print(f"Date range       : {df['date'].min()}  ->  {df['date'].max()}")
df[["participant_id", "date", "time", "latitude", "longitude", "speed", "speed_accuracy", "label"]].head()

## 1.2 All participant IDs

In [ ]:
participant_ids = sorted(df["participant_id"].unique())
print(f"Total participants: {len(participant_ids)}\n")
pd.DataFrame({"#": range(1, len(participant_ids) + 1), "participant_id": participant_ids}).set_index("#")

## 1.3 & 1.4 GPS points and days captured — per participant
* **`n_gps_points`** = number of GPS measurements recorded for that participant (rows in the file)
* **`n_days_with_gps`** = number of distinct calendar days on which at least one GPS point was recorded
* **`avg_points_per_day`** = `n_gps_points / n_days_with_gps`

In [ ]:
participant_summary = (
    df.groupby("participant_id")
      .agg(n_gps_points=("measurement_id", "count"),
           n_days_with_gps=("date", "nunique"),
           first_date=("date", "min"),
           last_date=("date", "max"))
      .reset_index()
)
participant_summary["avg_points_per_day"] = (participant_summary["n_gps_points"] / participant_summary["n_days_with_gps"]).round(1)
participant_summary.to_csv("gps_points_and_days_per_participant.csv", index=False)

print(f"Total GPS points across all participants: {participant_summary['n_gps_points'].sum():,}")
participant_summary

## 1.5 Day-wise detail: GPS points per day and first GPS timestamp
One row per **participant × date**:

| column | meaning |
|---|---|
| `participant_id` | participant |
| `date` | calendar day |
| `n_gps_points` | GPS points measured that day |
| `first_gps_time` | time of the first GPS point of that day |

The full table is also saved as **`gps_points_per_participant_per_day.csv`**.

In [ ]:
daily_gps = (
    df.groupby(["participant_id", "date"])
      .agg(n_gps_points=("measurement_id", "count"),
           first_gps_ts=("timestamp", "min"))
      .reset_index()
      .sort_values(["participant_id", "date"])
)
daily_gps["first_gps_time"] = daily_gps["first_gps_ts"].dt.strftime("%H:%M:%S")
daily_gps = daily_gps[["participant_id", "date", "n_gps_points", "first_gps_time"]].reset_index(drop=True)

daily_gps.to_csv("gps_points_per_participant_per_day.csv", index=False)
print(f"{len(daily_gps)} participant-day rows saved to gps_points_per_participant_per_day.csv")

with pd.option_context("display.max_rows", None):
    display(daily_gps)

## 1.6 Outlier detection & cleaning
Four checks are retained, but **coordinate cleaning is separated from diagnostic speed flags**.

| Flag | Meaning | Removes GPS row? |
|---|---|---|
| `flag_range` | impossible/zero latitude or longitude | **Yes** |
| `flag_iqr` | participant-specific latitude/longitude IQR outlier | **Yes** |
| `flag_speed` | missing/sentinel or implausibly high reported speed | No; diagnostic only |
| `flag_speed_accuracy` | missing/sentinel or poor reported speed accuracy | No; diagnostic only |

`is_location_outlier = flag_range OR flag_iqr` is the mask used to create the cleaned coordinate dataset. This avoids deleting a valid location merely because its speed metadata is poor. The cleaned CSV contains **only retained rows**; coordinates are never replaced by `0` or artificial `NaN` values.

### 1.6.1 Helper functions


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Distance in metres between two lat/long points (kept from original code; not used in the flags)."""
    R = 6_371_000
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlam = radians(lon2 - lon1)
    a = sin(dphi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(dlam / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


# STEP 1: hard range check
def flag_invalid_range(df):
    return (
        (df["latitude"]  < -90)  | (df["latitude"]  > 90) |
        (df["longitude"] < -180) | (df["longitude"] > 180) |
        (df["latitude"]  == 0)   | (df["longitude"] == 0)
    )


# STEP 2: IQR outliers, computed separately for each participant
def flag_iqr_outliers(group, multiplier=IQR_MULTIPLIER):
    outlier = pd.Series(False, index=group.index)
    for col in ["latitude", "longitude"]:
        Q1, Q3 = group[col].quantile(0.25), group[col].quantile(0.75)
        IQR = Q3 - Q1
        outlier |= (group[col] < Q1 - multiplier * IQR) | (group[col] > Q3 + multiplier * IQR)
    return outlier


# STEP 3: speed based checks (two sub-flags)
def flag_speed_outliers(df, max_speed_ms=MAX_SPEED_MS):
    """Speed sentinel (-1) OR physically implausible speed."""
    return (df["speed"] == -1) | (df["speed"] > max_speed_ms)

def flag_speed_accuracy_outliers(df, max_accuracy_ms=MAX_SPEED_ACCURACY):
    """Speed-accuracy sentinel (-1) OR poor GPS fix."""
    return (df["speed_accuracy"] == -1) | (df["speed_accuracy"] > max_accuracy_ms)

### 1.6.2 Apply all flags

In [ ]:
df["flag_range"] = flag_invalid_range(df)
df["flag_iqr"] = pd.concat([flag_iqr_outliers(g) for _, g in df.groupby("participant_id", sort=False)]).reindex(df.index).fillna(False).astype(bool)
df["flag_speed"] = flag_speed_outliers(df)
df["flag_speed_accuracy"] = flag_speed_accuracy_outliers(df)

# Coordinate outliers are the ONLY rows removed from the cleaned GPS dataset.
location_flag_cols = ["flag_range", "flag_iqr"]
diagnostic_flag_cols = ["flag_speed", "flag_speed_accuracy"]
all_flag_cols = location_flag_cols + diagnostic_flag_cols

df["is_location_outlier"] = df[location_flag_cols].any(axis=1)
df["is_any_flag"] = df[all_flag_cols].any(axis=1)
df["methods_flagged"] = df[all_flag_cols].sum(axis=1)
df["is_overlap"] = df["methods_flagged"] >= 2

# Backward-compatible name, now correctly means a LOCATION outlier.
df["is_outlier"] = df["is_location_outlier"]

print("Flags applied.")
print(f"Location outliers to remove: {df['is_location_outlier'].sum():,}")
print(f"Rows with any diagnostic/location flag: {df['is_any_flag'].sum():,}")


### 1.6.3 Global outlier breakdown
How many points each method flagged across **all** participants.

In [ ]:
print("=" * 60)
print("GLOBAL OUTLIER BREAKDOWN")
print("=" * 60)
print(f"  Invalid coordinate range : {df['flag_range'].sum():,}")
print(f"  Coordinate IQR           : {df['flag_iqr'].sum():,}")
print(f"  LOCATION rows removed    : {df['is_location_outlier'].sum():,} / {len(df):,} ({df['is_location_outlier'].mean()*100:.2f}%)")
print(f"  Location rows retained   : {(~df['is_location_outlier']).sum():,}")
print()
print("Diagnostic only (do NOT remove coordinates):")
print(f"  Speed sentinel/high      : {df['flag_speed'].sum():,}")
print(f"  Poor speed accuracy      : {df['flag_speed_accuracy'].sum():,}")
print(f"  Any flag                 : {df['is_any_flag'].sum():,}")


### 1.6.4 Pairwise overlap matrix (global)
Each cell = number of points flagged by **both** the row method and the column method. The diagonal is the total for that method.

In [ ]:
flag_labels = {"flag_range": "Range", "flag_iqr": "IQR", "flag_speed": "Speed", "flag_speed_accuracy": "SpeedAcc"}
overlap_matrix = pd.DataFrame(index=flag_labels.values(), columns=flag_labels.values(), dtype=int)
for c1, l1 in flag_labels.items():
    for c2, l2 in flag_labels.items():
        overlap_matrix.loc[l1, l2] = int((df[c1] & df[c2]).sum())
overlap_matrix

### 1.6.5 ⭐ Per-participant outlier summary
| column | meaning |
|---|---|
| `total_points` | all GPS points of the participant |
| `outlier_count` / `outlier_pct` | points flagged by at least one method, and as % of total |
| `range_outliers`, `iqr_outliers`, `speed_outliers`, `speed_accuracy_outliers` | how many points each individual method flagged |
| `overlap_points` | points flagged by 2+ methods |
| `overlap_pct` | overlap points as % of that participant's outliers |

In [ ]:
outlier_summary = df.groupby("participant_id").agg(
    total_points=("measurement_id", "count"),
    location_outlier_count=("is_location_outlier", "sum"),
    range_outliers=("flag_range", "sum"),
    iqr_outliers=("flag_iqr", "sum"),
    speed_flags=("flag_speed", "sum"),
    speed_accuracy_flags=("flag_speed_accuracy", "sum"),
    overlap_points=("is_overlap", "sum"),
).reindex(participant_ids).reset_index()

outlier_summary["location_outlier_pct"] = (outlier_summary["location_outlier_count"] / outlier_summary["total_points"] * 100).round(2)
outlier_summary["clean_points"] = outlier_summary["total_points"] - outlier_summary["location_outlier_count"]

assert len(outlier_summary) == 30, f"Expected 30 participants, got {len(outlier_summary)}"
assert set(outlier_summary["participant_id"]) == set(participant_ids)

with pd.option_context("display.max_rows", None):
    display(outlier_summary)


### 1.6.6 Per-participant pairwise overlap counts
For each participant: how many points were flagged by both methods of each pair.

In [ ]:
pairs = [
    ("flag_range", "flag_iqr",            "Range & IQR"),
    ("flag_range", "flag_speed",          "Range & Speed"),
    ("flag_range", "flag_speed_accuracy", "Range & SpeedAcc"),
    ("flag_iqr",   "flag_speed",          "IQR & Speed"),
    ("flag_iqr",   "flag_speed_accuracy", "IQR & SpeedAcc"),
    ("flag_speed", "flag_speed_accuracy", "Speed & SpeedAcc"),
]
overlap_records = []
for pid, grp in df.groupby("participant_id"):
    row = {"participant_id": pid}
    for c1, c2, label in pairs:
        row[label] = int((grp[c1] & grp[c2]).sum())
    overlap_records.append(row)

overlap_detail = pd.DataFrame(overlap_records)
with pd.option_context("display.max_rows", None):
    display(overlap_detail)

### 1.6.7 Remove coordinate-outlier rows and save
The cleaned file now contains **only genuine retained GPS rows**. No latitude/longitude is replaced with `0` or with a cleaning-generated `NaN`.

The original flagged dataset is also saved separately for auditing, so every removed measurement can be inspected.


In [ ]:
# Keep an audit copy with every original row + flags.
df.to_csv("gps_with_outlier_flags.csv", index=False)

# TRUE cleaning: remove coordinate-outlier rows instead of changing their coordinates.
clean = df.loc[~df["is_location_outlier"]].copy()

# Guard against pre-existing missing/invalid coordinates in the source data.
valid_coord = (
    clean["latitude"].notna() & clean["longitude"].notna() &
    clean["latitude"].between(-90, 90) & clean["longitude"].between(-180, 180) &
    clean["latitude"].ne(0) & clean["longitude"].ne(0)
)
clean = clean.loc[valid_coord].copy()

clean.to_csv("gps_outliers_removed.csv", index=False)
outlier_summary.to_csv("outlier_summary_per_participant.csv", index=False)
overlap_detail.to_csv("overlap_detail_per_participant.csv", index=False)

print(f"Original rows : {len(df):,}")
print(f"Clean rows    : {len(clean):,}")
print(f"Rows removed  : {len(df)-len(clean):,}")
print(f"Participants in source : {df['participant_id'].nunique()}")
print(f"Participants with >=1 clean GPS row: {clean['participant_id'].nunique()}")
print("Clean CSV contains no artificial zero/NaN coordinate replacement.")


## 1.7 Geomap of cleaned GPS points — one status/map per participant
Uses the truly filtered `gps_outliers_removed.csv`. The loop is driven by the **original 30 participant IDs**, so nobody silently disappears. A participant with zero retained coordinates is explicitly reported.

### 1.7.1 Load and validate the cleaned data


In [ ]:
clean = pd.read_csv(CLEANED_FILE_FOR_MAP, low_memory=False)

# Validate rather than silently dropping bad values.
assert clean[["latitude", "longitude"]].notna().all().all(), "Clean CSV still contains missing coordinates"
assert clean["latitude"].between(-90, 90).all() and clean["longitude"].between(-180, 180).all()
assert clean["latitude"].ne(0).all() and clean["longitude"].ne(0).all()

clean_ids = set(clean["participant_id"].unique())
missing_after_cleaning = [pid for pid in participant_ids if pid not in clean_ids]
print(f"Rows in clean file: {len(clean):,}")
print(f"Participants represented by clean GPS rows: {len(clean_ids)} / {len(participant_ids)}")
if missing_after_cleaning:
    print("Participants with 0 clean GPS points:", missing_after_cleaning)
else:
    print("All 30 participants retain at least one clean GPS point.")


### 1.7.2 Variability of the remaining points (per participant)
Quick numbers before the maps: how many points are left per label, and how spread out the points are
(`lat_std` / `lon_std` = standard deviation of coordinates in degrees; roughly, 0.01° ≈ 1.1 km).

In [ ]:
label_counts = clean.pivot_table(index="participant_id", columns="label", values="measurement_id", aggfunc="count", fill_value=0)
label_counts = label_counts.reindex(index=participant_ids, columns=list(LABEL_COLORS), fill_value=0)
label_counts["total_clean_points"] = label_counts.sum(axis=1)

spread = clean.groupby("participant_id").agg(
    lat_std=("latitude", "std"), lon_std=("longitude", "std"),
    lat_min=("latitude", "min"), lat_max=("latitude", "max"),
    lon_min=("longitude", "min"), lon_max=("longitude", "max"),
).reindex(participant_ids).round(4)

variability = label_counts.join(spread).reset_index()
assert len(variability) == 30
with pd.option_context("display.max_rows", None):
    display(variability)


### 1.7.3 The maps

In [ ]:
# Use the newer plotly map function if available, otherwise the older mapbox one (both use free OpenStreetMap tiles)
scatter_fn = getattr(px, "scatter_map", None) or px.scatter_mapbox
map_style_arg = {"map_style": "open-street-map"} if hasattr(px, "scatter_map") else {"mapbox_style": "open-street-map"}


def zoom_for(lat_span, lon_span):
    """Rough zoom level so the whole participant track fits in the map."""
    span = max(lat_span, lon_span, 1e-4)
    return float(np.clip(np.log2(360 / span) - 0.5, 1, 15))


def plot_participant_map(pid, data):
    d = data[data["participant_id"] == pid]
    if d.empty:
        print(f"Participant {pid}: no valid points left after cleaning")
        return
    counts = d["label"].value_counts()
    print(f"Participant {pid}  |  {len(d):,} points  |  " +
          "  ".join(f"{k}: {counts.get(k, 0):,}" for k in LABEL_COLORS))

    fig = scatter_fn(
        d, lat="latitude", lon="longitude", color="label",
        color_discrete_map=LABEL_COLORS, category_orders={"label": list(LABEL_COLORS)},
        hover_data={"date": True, "time": True, "latitude": ":.5f", "longitude": ":.5f", "label": False},
        opacity=0.7, height=550,
        center={"lat": d["latitude"].median(), "lon": d["longitude"].median()},
        zoom=zoom_for(d["latitude"].max() - d["latitude"].min(), d["longitude"].max() - d["longitude"].min()),
        title=f"Participant {pid} - cleaned GPS points by label",
        **map_style_arg,
    )
    fig.update_traces(marker={"size": 6})
    fig.update_layout(margin={"l": 0, "r": 0, "t": 40, "b": 0}, legend_title_text="Label")
    fig.show()


ids_to_plot = PARTICIPANTS_TO_PLOT if PARTICIPANTS_TO_PLOT else participant_ids
for pid in ids_to_plot:
    plot_participant_map(pid, clean)

---
# PART 2 — EMA DATA
---
## 2.1 What counts as an EMA record in this file?
The EMA (emotion) scores in this file are **daily values**: `pos_emotion` and `neg_emotion` (plus derived columns such as
`*_std`, `*_baseline`, `*_z`) have **exactly one value per participant per day**, repeated on every GPS row of that day.
So the notebook first collapses the file to **one row per participant × day** and counts an EMA as **"done" when the EMA score columns
(`EMA_SCORE_COLS`) are filled** for that day.

⚠️ **Important limitation:** the file contains **no individual beep / EMA timestamps** (the `time`, `datetime` columns belong to the
GPS points). Therefore the individual EMA *times* cannot be extracted from this file. Section 2.3 is already prepared: if you later
have a file with a beep-time column, set `EMA_TIMESTAMP_COL` in the settings cell and the times will be listed as `HH:MM, HH:MM, ...`.

In [ ]:
# Check: does every EMA score column have one single value per participant-day?
check = df.groupby(["participant_id", "date"])[EMA_SCORE_COLS].nunique()
print("Max number of distinct values per participant-day (1 = one daily value):")
print(check.max().to_string())
print()
print("Columns in the file that could hold a beep time:",
      [c for c in df.columns if any(k in c.lower() for k in ["ema", "beep", "survey", "response"])] or "none")

# One row per participant-day that has EMA scores recorded
ema_daily = (
    df.dropna(subset=EMA_SCORE_COLS)
      .sort_values("timestamp")
      .drop_duplicates(["participant_id", "date"])
      [["participant_id", "date"] + EMA_SCORE_COLS]
      .sort_values(["participant_id", "date"])
      .reset_index(drop=True)
)
print(f"\nParticipant-days with EMA recorded: {len(ema_daily)}")

## 2.2 EMA count and number of EMA days — per participant
* **`n_ema_recorded`** = number of EMA records (daily score sets) for the participant
* **`n_days_with_ema`** = number of distinct days with an EMA
* **`n_days_with_gps`** = days with GPS (for comparison)
* **`days_gps_without_ema`** = GPS days on which no EMA was recorded

In [ ]:
ema_summary = (
    ema_daily.groupby("participant_id")
             .agg(n_ema_recorded=("date", "count"),
                  n_days_with_ema=("date", "nunique"),
                  first_ema_date=("date", "min"),
                  last_ema_date=("date", "max"))
             .reset_index()
)
ema_summary = participant_summary[["participant_id", "n_days_with_gps"]].merge(ema_summary, on="participant_id", how="left").fillna({"n_ema_recorded": 0, "n_days_with_ema": 0})
ema_summary["days_gps_without_ema"] = ema_summary["n_days_with_gps"] - ema_summary["n_days_with_ema"]
ema_summary[["n_ema_recorded", "n_days_with_ema"]] = ema_summary[["n_ema_recorded", "n_days_with_ema"]].astype(int)
assert len(ema_summary) == 30 and set(ema_summary["participant_id"]) == set(participant_ids)
ema_summary.to_csv("ema_summary_per_participant.csv", index=False)

print(f"Total EMA records across all participants: {ema_summary['n_ema_recorded'].sum()}")
ema_summary

## 2.3 Day-wise EMA detail — per participant, per day
One row per **participant × date**:

| column | meaning |
|---|---|
| `n_ema_that_day` | number of EMA records that day |
| `ema_times_HHMM` | times of the EMAs that day as `HH:MM, HH:MM, ...` — filled only if `EMA_TIMESTAMP_COL` is set (see 2.1) |
| `pos_emotion`, `neg_emotion` | the EMA scores of that day |

Saved as **`ema_per_participant_per_day.csv`**.

In [ ]:
def format_times(series):
    """Turn several timestamps into '08:15, 12:40, 19:05' (hour and minute only)."""
    t = pd.to_datetime(series, format="mixed", errors="coerce").dropna().sort_values()
    return ", ".join(t.dt.strftime("%H:%M"))


if EMA_TIMESTAMP_COL:
    raw = df.dropna(subset=[EMA_TIMESTAMP_COL]).drop_duplicates(["participant_id", "date", EMA_TIMESTAMP_COL])
    times = raw.groupby(["participant_id", "date"])[EMA_TIMESTAMP_COL].apply(format_times).rename("ema_times_HHMM").reset_index()
    n_beeps = raw.groupby(["participant_id", "date"]).size().rename("n_ema_that_day").reset_index()
    ema_by_day = ema_daily.merge(n_beeps, on=["participant_id", "date"], how="left").merge(times, on=["participant_id", "date"], how="left")
else:
    ema_by_day = ema_daily.copy()
    ema_by_day["n_ema_that_day"] = 1                                   # one daily EMA record
    ema_by_day["ema_times_HHMM"] = "not available in this file"        # no beep timestamps exist

ema_by_day = ema_by_day[["participant_id", "date", "n_ema_that_day", "ema_times_HHMM"] + EMA_SCORE_COLS]
ema_by_day.to_csv("ema_per_participant_per_day.csv", index=False)
print(f"{len(ema_by_day)} participant-day rows saved to ema_per_participant_per_day.csv")

with pd.option_context("display.max_rows", None):
    display(ema_by_day)

---
## ✅ Files created by this notebook
| File | Content |
|---|---|
| `gps_points_and_days_per_participant.csv` | GPS points & days; all 30 participants |
| `gps_points_per_participant_per_day.csv` | participant-day GPS detail |
| `outlier_summary_per_participant.csv` | coordinate-removal + diagnostic flag summary; all 30 participants |
| `overlap_detail_per_participant.csv` | pairwise flag overlap |
| `gps_with_outlier_flags.csv` | every original GPS row plus audit flags |
| `gps_outliers_removed.csv` | **true filtered GPS data; outlier rows removed, no 0/NaN substitution** |
| `ema_summary_per_participant.csv` | EMA summary; all 30 participants |
| `ema_per_participant_per_day.csv` | EMA per participant-day |
